# 🎬 Viral Shorts AI — FREE Colab Edition

Lambi video do → AI behtareen moments dhoondhe → **Vizard jaisi** 9:16 vertical clips
bana de (speaker ka chehra track karke, karaoke captions ke saath). **Bilkul muft** —
Google Colab ka free GPU istemal hota hai.

**Tareeqa (phone se bhi):**
1. Upar **Runtime → Run all** dabao (pehli dafa 3-5 minute lagega — AI models download honge).
2. Jab kaha jaye, apni **video upload** karo.
3. Neeche settings me clips ki tadad/lambai chuno.
4. Aakhir me clips **download** kar lo — YouTube Shorts / TikTok / Reels ke liye tayyar!

⚠️ **Imandaar notes:** Muft Colab GPU kabhi kabhi busy hota hai (thoda intezar karna par sakta hai).
Agar session band ho jaye to Runtime → Run all dobara dabao. Pehli dafa model download
me waqt lagta hai, uske baad tez ho jata hai.


## STEP 0 — Setup (sirf ek dafa chalao)


In [ ]:
import subprocess, sys

print('🔍 GPU check...')
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                   capture_output=True, text=True)
if r.returncode == 0:
    print('✅ GPU mil gaya:', r.stdout.strip())
else:
    print('⚠️ GPU NAHI mila! Upar Runtime → Change runtime type → Hardware accelerator = GPU (T4) chuno, phir dobara chalao.')

print('\n📦 Packages install ho rahe hain (2-3 minute)...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'faster-whisper', 'mediapipe', 'opencv-python-headless', 'numpy'], check=True)
r = subprocess.run(['which', 'ffmpeg'], capture_output=True, text=True)
print('✅ ffmpeg maujood hai' if r.returncode == 0 else '❌ ffmpeg nahi mila')
print('\n🎉 Setup mukammal!')


## STEP 1 — Video upload karo


In [ ]:
from google.colab import files
import shutil, os

print('📁 Apni lambi video chuno (MP4 behtar hai)...')
uploaded = files.upload()
if not uploaded:
    raise SystemExit('❌ Koi file upload nahi hui — ye cell dobara chalao.')
src_name = list(uploaded.keys())[0]
shutil.copy(src_name, '/content/source.mp4')
mb = os.path.getsize('/content/source.mp4') / 1048576
print(f'✅ Video mil gayi: {src_name} ({mb:.1f} MB)')


## STEP 2 — Settings (apni marzi se badlo)


In [ ]:
NUM_CLIPS = 3  #@param {type:"slider", min:1, max:8, step:1}
CLIP_SECONDS = 45  #@param {type:"slider", min:15, max:90, step:5}
LANGUAGE = "auto"  #@param ["auto", "en", "ur", "hi"] {type:"string"}
WHISPER_MODEL = "medium"  #@param ["tiny", "base", "small", "medium"] {type:"string"}
CAPTIONS_ON = True  #@param {type:"boolean"}

print(f'⚙️ {NUM_CLIPS} clips × {CLIP_SECONDS} second | Zaban: {LANGUAGE} | AI model: {WHISPER_MODEL} | Captions: {"on" if CAPTIONS_ON else "off"}')
print('ℹ️ Model bara = behtar captions, lekin pehli dafa download me zyada waqt.')


## STEP 3 — AI kaam (transcribe → best moments → clips)


In [ ]:
# ---------- Audio ----------
import subprocess, wave, numpy as np, shutil, re, math, logging
logging.basicConfig(level=logging.WARNING)

SRC = '/content/source.mp4'
WORK = '/content/work'
import os; os.makedirs(WORK, exist_ok=True)

def extract_audio(video_path, out_wav):
    cmd = ['ffmpeg', '-y', '-i', video_path, '-vn', '-ac', '1', '-ar', '16000',
           '-c:a', 'pcm_s16le', out_wav]
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.returncode != 0: raise RuntimeError('Audio nikalne me masla: ' + p.stderr[-500:])
    return out_wav

def audio_energy(wav_path, frame_sec=0.5):
    wf = wave.open(wav_path, 'rb'); nch, sw, fr = wf.getnchannels(), wf.getsampwidth(), wf.getframerate()
    raw = wf.readframes(wf.getnframes()); wf.close()
    audio = np.frombuffer(raw, dtype={1: np.int8, 2: np.int16, 4: np.int32}[sw]).astype(np.float32)
    audio /= float(2 ** (8 * sw - 1))
    if nch > 1: audio = audio.reshape(-1, nch).mean(axis=1)
    fl = max(1, int(frame_sec * fr)); out = []
    for s in range(0, len(audio), fl):
        ch = audio[s:s+fl]
        if ch.size == 0: break
        out.append((round(s / fr, 3), float(np.sqrt(np.mean(ch ** 2)))))
    return out

# ---------- Transcribe (faster-whisper, GPU) ----------
def transcribe_audio(audio_path, model_size, language):
    import torch
    from faster_whisper import WhisperModel
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    compute = 'float16' if device == 'cuda' else 'int8'
    print(f'🧠 AI sun raha hai... (model={model_size}, device={device})')
    model = WhisperModel(model_size, device=device, compute_type=compute)
    lang = None if language == 'auto' else language
    segs, _ = model.transcribe(audio_path, language=lang, word_timestamps=True, vad_filter=True)
    segments, words = [], []
    for s in segs:
        segments.append({'start': float(s.start), 'end': float(s.end), 'text': s.text.strip()})
        for w in (s.words or []):
            words.append({'word': w.word.strip(), 'start': float(w.start), 'end': float(w.end)})
    print(f'✅ Suna liya: {len(segments)} jumle, {len(words)} alfaaz')
    return segments, words


In [ ]:
# ---------- Best moments (viral scoring) ----------
WEIGHTS = {'hook': .25, 'curiosity': .15, 'emotion': .15, 'info': .15, 'audio': .10, 'question': .10, 'completion': .10}
HOOK_WORDS = frozenset('stop secret secrets never nobody mistake mistakes shocking insane crazy free truth lie lies hack hacks warning dangerous proven guaranteed million billion viral banned hidden finally urgent breaking exposed revealed forbidden'.split())
OPEN_LOOPS = ('watch till the end','wait for it','wait until the end','stay tuned','what happens next',"you won't believe",'here is why',"here's why",'but first','the truth is','nobody tells you','in the end','at the end','keep watching',"don't skip")
CONTRAST = frozenset('but however although though yet instead until'.split())
EMO = {'love':1,'hate':1,'amazing':.9,'terrible':.9,'shocking':.9,'insane':.8,'crazy':.8,'unbelievable':.9,'incredible':.9,'scary':.8,'angry':.8,'furious':.9,'sad':.7,'happy':.7,'excited':.8,'worried':.7,'proud':.7,'surprised':.8,'beautiful':.7,'horrible':.9,'brilliant':.8,'genius':.8,'disaster':.9,'miracle':.9,'nightmare':.9,'epic':.8,'wild':.7}
WH = frozenset('who what when where why how which'.split())
SENT_END = re.compile(r'[.!?…]+$'); WRE = re.compile(r"[A-Za-z0-9']+")

def sentences_of(segs):
    out, buf, st = [], [], None
    for s in segs:
        t = (s.get('text') or '').strip()
        if not t: continue
        if st is None: st = float(s['start'])
        buf.append(s)
        if SENT_END.search(t):
            out.append({'start': st, 'end': float(buf[-1]['end']), 'text': ' '.join(x['text'].strip() for x in buf)}); buf, st = [], None
    if buf: out.append({'start': st, 'end': float(buf[-1]['end']), 'text': ' '.join(x['text'].strip() for x in buf)})
    return out

def gen_candidates(segs, durations, tol=2.0, min_gap=5.0):
    snts = sentences_of(segs); cands = []
    for target in sorted({int(d) for d in durations if d and d > 0}):
        for i in range(len(snts)):
            st = snts[i]['start']; j = i
            while j < len(snts) and snts[j]['end'] - st <= target + tol: j += 1
            if j == i: continue
            en = snts[j-1]['end']
            if en - st >= target - tol: cands.append({'start': round(st,3), 'end': round(en,3), 'duration': round(en-st,3)})
    cands.sort(key=lambda c: (c['start'], c['end'])); uniq = []
    for c in cands:
        if not any(abs(c['start']-u['start'])<1 and abs(c['end']-u['end'])<1 for u in uniq): uniq.append(c)
    spaced = []
    for c in uniq:
        if all(abs(c['start']-s['start']) >= min_gap for s in spaced): spaced.append(c)
    return spaced

def score_cand(c, segs, words, energy):
    ww = [w for w in words if w['start'] >= c['start']-1e-6 and w['end'] <= c['end']+1e-6]
    text = ' '.join(s.get('text','') for s in segs if s['end']>c['start'] and s['start']<c['end']).strip()
    low = text.lower(); toks = WRE.findall(low); n = len(toks); dur = max(c['end']-c['start'], 1e-6)
    f5 = ' '.join(w['word'] for w in ww if w['start'] < c['start']+5); f5t = WRE.findall(f5.lower())
    hook = min(1, sum(1 for t in f5t if t in HOOK_WORDS)*.25 + ('?' in f5)*.45 + bool(re.search(r'\d', f5))*.2)
    cur = min(1, sum(1 for p in OPEN_LOOPS if p in low)*.45 + sum(1 for t in toks if t in CONTRAST)*.15)
    emo = min(1, sum(EMO.get(t,0) for t in toks)/3)
    wps = n/dur; info = .6*max(0, 1-abs(wps-2.5)/2.5) + .4*min(1, len(re.findall(r'\b([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)\b', text))*.25)
    wr = [r for (t,r) in energy if c['start']<=t<=c['end']]
    audio = min(1, (sum(wr)/len(wr))/(sum(r for _,r in energy)/len(energy)+1e-9)/1.4) if wr and energy else .5
    q = 1.0 if '?' in text else (0.6 if any(t in WH for t in toks) else 0.0)
    comp = 1.0 if SENT_END.search(text) else .4
    if ww:
        if c['end']-ww[-1]['end'] > 1.5: comp *= .5
        if ww[0]['start']-c['start'] > 1.0: comp *= .8
    total = sum(WEIGHTS[k]*v for k,v in {'hook':hook,'curiosity':cur,'emotion':emo,'info':info,'audio':audio,'question':q,'completion':comp}.items())*100
    return round(total, 1)

def pick_top(scored, n, max_overlap=0.3):
    ranked = sorted(scored, key=lambda c: c['score'], reverse=True); chosen = []
    for c in ranked:
        if all(max(0, min(c['end'],s['end'])-max(c['start'],s['start'])) <= max_overlap*min(c['duration'],s['duration']) for s in chosen):
            chosen.append(c)
        if len(chosen) >= n: break
    return chosen

# ---------- Chalao ----------
print('🎧 Audio nikal rahe hain...')
wav = extract_audio(SRC, WORK + '/audio.wav')
energy = audio_energy(wav)
segments, words = transcribe_audio(wav, WHISPER_MODEL, LANGUAGE)
if not words: raise SystemExit('❌ Video me boli hui awaz nahi mili — koi aur video try karo.')
print('🎯 Best moments dhoond rahe hain...')
cands = gen_candidates(segments, [CLIP_SECONDS])
for c in cands: c['score'] = score_cand(c, segments, words, energy)
top = pick_top(cands, NUM_CLIPS)
if not top: raise SystemExit('❌ Koi acha moment nahi mila — CLIP_SECONDS chhota karke dobara chalao.')
for i, c in enumerate(top, 1):
    print(f'  ✅ Clip {i}: {c["start"]:.0f}s → {c["end"]:.0f}s  (score {c["score"]})')
print(f'\n🎉 {len(top)} clips mil gaye! Agla cell chalao →')


## STEP 4 — Clips banana (face tracking + 9:16 + captions) — sab se lamba step


In [ ]:
# ---------- Face tracking (MediaPipe) ----------
import cv2
try:
    import mediapipe as mp
    _det = mp.solutions.face_detection.FaceDetection(model_selection=0, min_detection_confidence=0.5)
    HAS_FACE = True; print('✅ Face tracking tayyar')
except Exception as e:
    HAS_FACE = False; print('⚠️ Face tracking nahi chal saka — center-crop istemal hoga:', str(e)[:100])

def detect_bgr(frame):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB); res = _det.process(rgb); out = []
    if res.detections:
        for d in res.detections:
            b = d.location_data.relative_bounding_box
            out.append((max(0.,float(b.xmin)), max(0.,float(b.ymin)), float(b.width), float(b.height)))
    return out

def _fscore(d):
    x,y,w,h = d; cx, cy = x+w/2, y+h/2
    cent = 1.0 - min(1.0, math.hypot(cx-0.5, cy-0.5)*1.2)
    return w*h*(0.5+0.5*cent)

def speaker_track(dets, alpha=0.35):
    track, state = {}, None
    for f in sorted(dets):
        if not dets[f]: continue
        x,y,w,h = max(dets[f], key=_fscore)
        cur = (x+w/2, y+h/2, w, h)
        state = cur if state is None else tuple(c*alpha+s*(1-alpha) for c,s in zip(cur,state))
        track[f] = state
    return track

# ---------- 9:16 crop plan ----------
def plan_boxes(track, sw, sh, nf):
    cw, ch = int(sh*9/16), sh
    if cw > sw: cw, ch = sw, int(sw*16/9)
    keys = sorted(track); boxes = []
    for f in range(nf):
        if keys and HAS_FACE:
            if f <= keys[0]: e = track[keys[0]]
            elif f >= keys[-1]: e = track[keys[-1]]
            else:
                e = track[keys[0]]
                for i in range(len(keys)-1):
                    a,b = keys[i], keys[i+1]
                    if a <= f <= b:
                        t=(f-a)/(b-a); va,vb=track[a],track[b]
                        e=tuple(x+(y-x)*t for x,y in zip(va,vb)); break
            cx,cy,_,hh = e; fx, fy = cx*sw, cy*sh
            x = fx-cw/2; y = (fy-hh*sh/2)-0.25*ch
        else:
            x = (sw-cw)/2; y = max(0., (sh-ch)/2-0.08*ch)
        x = min(max(0.,x), sw-cw); y = min(max(0.,y), sh-ch)
        boxes.append((int(round(x)), int(round(y)), cw, ch))
    return boxes

# ---------- Karaoke captions (ASS) ----------
def ass_ts(s):
    s=max(0.,s); return f'{int(s//3600)}:{int(s%3600//60):02d}:{s%60:05.2f}'
def build_ass(words, cs, ce):
    win = sorted([w for w in words if w['end']>cs and w['start']<ce], key=lambda w: w['start'])
    evs = []
    for i in range(0, len(win), 16):
        ch = win[i:i+16]
        def kl(ws): return ' '.join('{\\k%d}%s' % (max(1,int(round((w['end']-w['start'])*100))), w['word'].replace('\\','\\\\').replace('{','\\{').replace('}','\\}').strip()) for w in ws)
        txt = kl(ch[:8]) + ('\\N'+kl(ch[8:16]) if len(ch)>8 else '')
        evs.append(f"Dialogue: 0,{ass_ts(ch[0]['start']-cs)},{ass_ts(ch[-1]['end']-cs)},Default,,0,0,0,,{txt}")
    head = '\n'.join(['[Script Info]','Title: Viral Shorts AI','ScriptType: v4.00+','WrapStyle: 0','ScaledBorderAndShadow: yes','PlayResX: 1080','PlayResY: 1920','','[V4+ Styles]','Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding','Style: Default,DejaVu Sans,86,&H00FFFFFF,&H0000FFFF,&H00000000,&H99000000,-1,0,0,0,100,100,0.5,0,1,5,1,2,40,40,170,1','','[Events]','Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text'])
    return head + '\n' + '\n'.join(evs) + '\n'

# ---------- Render ----------
import glob
cap = cv2.VideoCapture(SRC)
FPS = cap.get(cv2.CAP_PROP_FPS) or 30; SW = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); SH = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f'🎞️ Source: {SW}x{SH} @ {FPS:.1f}fps')
os.makedirs('/content/clips', exist_ok=True)
made = []
for ci, cl in enumerate(top, 1):
    st, en = cl['start'], cl['end']; dur = en-st; nf = int(dur*FPS)
    print(f'\n▶️ Clip {ci}/{len(top)} ({dur:.0f}s) — chehre dhoond rahe hain...')
    dets = {}
    if HAS_FACE:
        for f in range(0, nf, 6):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(st*FPS)+f)
            ok, fr = cap.read()
            if ok: dets[f] = detect_bgr(fr)
    track = speaker_track(dets) if dets else {}
    print('   ✅ Track:', 'speaker mil gaya 🎯' if track else 'center-crop (chehra nahi mila)')
    boxes = plan_boxes(track, SW, SH, nf)
    print('   🎨 Frames kaat rahe hain (9:16)...')
    raw = f'{WORK}/clip{ci}_raw.mp4'
    vw = cv2.VideoWriter(raw, cv2.VideoWriter_fourcc(*'mp4v'), FPS, (1080,1920))
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(st*FPS))
    for i in range(nf):
        ok, fr = cap.read()
        if not ok: break
        x,y,w,h = boxes[i]
        vw.write(cv2.resize(fr[y:y+h, x:x+w], (1080,1920)))
    vw.release()
    ass_p = f'{WORK}/clip{ci}.ass'; out = f'/content/clips/short_{ci:02d}.mp4'
    vf = []
    if CAPTIONS_ON and words:
        open(ass_p, 'w', encoding='utf-8-sig').write(build_ass(words, st, en))
        vf = ['-vf', f'ass={ass_p}']
        print('   ✍️ Captions lag rahe hain...')
    cmd = ['ffmpeg','-y','-i',raw,'-ss',str(st),'-t',str(dur),'-i',SRC,
           '-map','0:v','-map','1:a','-c:v','libx264','-preset','veryfast','-crf','20',
           '-pix_fmt','yuv420p','-c:a','aac','-b:a','128k'] + vf + [out]
    pr = subprocess.run(cmd, capture_output=True, text=True)
    if pr.returncode != 0: raise RuntimeError('Render nakam: '+pr.stderr[-600:])
    made.append(out); print(f'   ✅ Tayyar: {out}')
cap.release()
print(f'\n🎉 SAB TAYYAR! {len(made)} clips — agla cell chalao download ke liye →')


## STEP 5 — Download karo 📥


In [ ]:
from google.colab import files as _f
from IPython.display import Video, display
import glob
for p in sorted(glob.glob('/content/clips/*.mp4')):
    print('🎬', p); display(Video(p, width=270, embed=True))
print('\n⬇️ Download shuru...')
for p in sorted(glob.glob('/content/clips/*.mp4')):
    _f.download(p)
print('✅ Ho gaya! Clips phone me download ho gaye.')
